In [ ]:
# ======================================================================================
# FINAL TARGETED PATCH — V7
#
# PURPOSE
# -------
# Patch ONLY the 14 re-annotated, previously clipped leads into:
#
#     final_digitisation_v6/X_gaze_final.npy
#
# using:
#
#     LOCAL_USER_HOME/Downloads/ecg_anchor_annotations_targeted*.json
#
# IMPORTANT IMPROVEMENTS
# ----------------------
# 1. Existing V6 dataset is the parent dataset.
# 2. ONLY the 14 targeted leads are replaced.
# 3. Expanded crops are used.
# 4. Manual anchors determine morphology.
# 5. Image snapping is local only (<= 6 px).
# 6. EVERY corrected lead is reconstructed over the FULL crop width.
# 7. No stretching of a partial manually annotated interval to 1000 samples.
# 8. QC explicitly checks crop-edge contact/clipping.
#
# OUTPUT
# ------
# final_digitisation_v7/
#
#     X_gaze_final.npy
#     FINAL_OVERLAY_<ECG>.png
#     FINAL_SIGNAL_<ECG>.png
#     targeted_patch_log.csv
#     targeted_patch_metrics.csv
#     final_array_qc.csv
#     metadata.json
#
# DO NOT TRAIN UNTIL THE FINAL_OVERLAY FILES HAVE BEEN INSPECTED.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import json
import hashlib
import warnings

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from scipy.ndimage import gaussian_filter
from scipy.interpolate import PchipInterpolator


warnings.filterwarnings("ignore")


# ======================================================================================
# 1. PATHS
# ======================================================================================

OUT_DIR = Path(
    "final_digitisation_v7"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Parent dataset containing all previously accepted V6 corrections
V6_ARRAY_PATH = Path(
    "final_digitisation_v6/X_gaze_final.npy"
)


GEOMETRY_PATH = Path(
    "manual_digitization_qc/manual_geometry.json"
)


DOWNLOADS = Path(
    "LOCAL_USER_HOME/Downloads"
)


# --------------------------------------------------------------------------------------
# Find newest TARGETED annotation JSON
# --------------------------------------------------------------------------------------

targeted_candidates = list(
    DOWNLOADS.glob(
        "ecg_anchor_annotations_targeted*.json"
    )
)


if len(targeted_candidates) == 0:

    raise FileNotFoundError(
        "\nCould not find:\n"
        "LOCAL_USER_HOME/Downloads/ecg_anchor_annotations_targeted*.json"
    )


TARGETED_JSON_PATH = max(
    targeted_candidates,
    key=lambda p: p.stat().st_mtime
)


print("=" * 100)
print("INPUTS")
print("=" * 100)

print(
    "Parent V6 array:",
    V6_ARRAY_PATH
)

print(
    "Manual geometry: ",
    GEOMETRY_PATH
)

print(
    "Targeted JSON:   ",
    TARGETED_JSON_PATH
)


for path in [
    V6_ARRAY_PATH,
    GEOMETRY_PATH,
    TARGETED_JSON_PATH,
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


# ======================================================================================
# 2. LOAD
# ======================================================================================

X_v6 = np.load(
    V6_ARRAY_PATH
).astype(
    np.float32
)


if X_v6.shape != (
    10,
    12,
    1000,
    1
):

    raise ValueError(
        f"Unexpected V6 shape: {X_v6.shape}"
    )


# Start with an exact copy of the accepted V6 dataset.
X_final = X_v6.copy()


with open(
    GEOMETRY_PATH,
    "r"
) as f:

    geometry = json.load(
        f
    )


with open(
    TARGETED_JSON_PATH,
    "r"
) as f:

    targeted_json = json.load(
        f
    )


annotations = targeted_json.get(
    "annotations",
    {}
)


if len(annotations) == 0:

    raise RuntimeError(
        "Targeted JSON contains no annotations."
    )


# ======================================================================================
# 3. FIXED ECG AND LEAD ORDERS
# ======================================================================================

ECG_ORDER = [

    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


LEAD_ORDER = [

    "I",
    "II",
    "III",

    "aVR",
    "aVL",
    "aVF",

    "V1",
    "V2",
    "V3",

    "V4",
    "V5",
    "V6",
]


DISPLAY_GRID = [

    ["I",   "aVR", "V1", "V4"],

    ["II",  "aVL", "V2", "V5"],

    ["III", "aVF", "V3", "V6"],
]


# ======================================================================================
# 4. EXACT SET OF LEADS THAT ARE ALLOWED TO CHANGE
# ======================================================================================

TARGET_LEADS = {

    "Hyperkalemia": [
        "V3",
    ],

    "VTach": [
        "II",
        "III",
        "aVF",
        "V2",
    ],

    "VentPaced": [
        "V2",
        "V3",
        "V4",
        "V5",
    ],

    "LBBB": [
        "V1",
        "V2",
        "V3",
    ],

    "CompleteHeartBlock": [
        "V1",
        "V2",
    ],
}


EXPECTED_KEYS = set()


for ecg, leads in TARGET_LEADS.items():

    for lead in leads:

        EXPECTED_KEYS.add(
            ecg
            +
            "::"
            +
            lead
        )


assert len(
    EXPECTED_KEYS
) == 14


# ======================================================================================
# 5. STRICTLY VERIFY TARGETED JSON
# ======================================================================================

summary_rows = []


for key, ann in annotations.items():

    # Ignore any stale browser localStorage entry that isn't one
    # of our actual 14 requested leads.
    if key not in EXPECTED_KEYS:

        continue


    summary_rows.append({

        "key":
            key,

        "ECG":
            ann.get(
                "ecg"
            ),

        "lead":
            ann.get(
                "lead"
            ),

        "status":
            ann.get(
                "status"
            ),

        "n_points":
            len(
                ann.get(
                    "points",
                    []
                )
            ),

        "first_fraction":
            ann.get(
                "horizontal_coverage",
                {}
            ).get(
                "first_fraction",
                np.nan
            ),

        "last_fraction":
            ann.get(
                "horizontal_coverage",
                {}
            ).get(
                "last_fraction",
                np.nan
            ),

        "span_fraction":
            ann.get(
                "horizontal_coverage",
                {}
            ).get(
                "span_fraction",
                np.nan
            ),
    })


summary_df = pd.DataFrame(
    summary_rows
)


print(
    "\n" +
    "=" * 100
)

print(
    "TARGETED ANNOTATION SUMMARY"
)

print(
    "=" * 100
)


print(
    summary_df
    .sort_values(
        [
            "ECG",
            "lead"
        ]
    )
    .to_string(
        index=False
    )
)


found_keys = set(
    summary_df[
        "key"
    ]
)


missing_keys = (
    EXPECTED_KEYS
    -
    found_keys
)


if len(
    missing_keys
) > 0:

    raise RuntimeError(
        "\nTargeted JSON is missing:\n"
        +
        "\n".join(
            sorted(
                missing_keys
            )
        )
    )


if len(
    summary_df
) != 14:

    raise RuntimeError(
        f"Expected 14 targeted annotations, found {len(summary_df)}."
    )


unfinished = summary_df[
    summary_df[
        "status"
    ]
    !=
    "corrected"
]


if len(
    unfinished
) > 0:

    print(
        "\nUNFINISHED TARGETS:"
    )

    print(
        unfinished.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "All 14 targeted leads must be marked corrected."
    )


too_few_points = summary_df[
    summary_df[
        "n_points"
    ]
    <
    10
]


if len(
    too_few_points
) > 0:

    raise RuntimeError(
        "At least one targeted lead has fewer than 10 anchors."
    )


# --------------------------------------------------------------------------------------
# Independently recalculate horizontal coverage from actual click coordinates later.
# Don't rely only on browser metadata.
# --------------------------------------------------------------------------------------

print(
    "\n✓ All 14 targeted annotations are present and marked corrected."
)


# ======================================================================================
# 6. SOURCE IMAGE HELPERS
# ======================================================================================

def resolve_source_image(
    stored_path
):

    p = Path(
        stored_path
    )


    if p.exists():

        return p


    matches = list(
        Path(".").glob(
            f"**/{p.name}"
        )
    )


    if len(
        matches
    ) == 0:

        raise FileNotFoundError(
            f"Cannot locate source image: {p.name}"
        )


    return matches[
        0
    ]


def load_gray(
    path
):

    return (
        np.asarray(
            Image.open(
                path
            )
            .convert(
                "L"
            )
        )
        .astype(
            np.float32
        )
        /
        255.0
    )


# ======================================================================================
# 7. MAP BROWSER POINTS BACK TO ORIGINAL EXPANDED CROP
# ======================================================================================

def browser_points_to_crop(
    ann
):

    points = np.asarray(

        [
            [
                float(
                    p[
                        "x"
                    ]
                ),
                float(
                    p[
                        "y"
                    ]
                )
            ]

            for p
            in ann[
                "points"
            ]
        ],

        dtype=np.float32
    )


    display_w = float(
        ann[
            "display_width"
        ]
    )


    display_h = float(
        ann[
            "display_height"
        ]
    )


    original_w = float(
        ann[
            "original_crop_width"
        ]
    )


    original_h = float(
        ann[
            "original_crop_height"
        ]
    )


    points[
        :,
        0
    ] *= (
        original_w
        /
        display_w
    )


    points[
        :,
        1
    ] *= (
        original_h
        /
        display_h
    )


    points = points[
        np.argsort(
            points[
                :,
                0
            ]
        )
    ]


    # ----------------------------------------------------------------------------------
    # Remove any nearly duplicated x positions after scale conversion.
    # ----------------------------------------------------------------------------------

    cleaned = [
        points[
            0
        ]
    ]


    for point in points[
        1:
    ]:

        if (
            point[
                0
            ]
            -
            cleaned[
                -1
            ][
                0
            ]
            >=
            0.50
        ):

            cleaned.append(
                point
            )


    points = np.asarray(
        cleaned,
        dtype=np.float32
    )


    if len(
        points
    ) < 8:

        raise RuntimeError(
            "Too few unique anchors after coordinate conversion."
        )


    return points


# ======================================================================================
# 8. LOCAL ECG INK SCORE
# ======================================================================================

def ecg_score_map(
    gray
):

    """
    ECG evidence score.

    No removal of vertical structures.
    No removal of horizontal structures.

    This preserves QRS complexes, paced spikes, VT morphology etc.
    """

    gray = np.asarray(
        gray,
        dtype=np.float32
    )


    p1 = np.percentile(
        gray,
        1
    )


    p99 = np.percentile(
        gray,
        99
    )


    if (
        p99
        >
        p1
        +
        1e-8
    ):

        norm = (
            gray
            -
            p1
        ) / (
            p99
            -
            p1
        )

    else:

        norm = gray.copy()


    norm = np.clip(
        norm,
        0,
        1
    )


    darkness = (
        1.0
        -
        norm
    )


    # Thin dark ridge relative to local background
    fine = gaussian_filter(
        norm,
        sigma=0.45
    )


    background = gaussian_filter(
        norm,
        sigma=2.2
    )


    ridge = np.clip(
        background
        -
        fine,
        0,
        None
    )


    ridge_scale = np.percentile(
        ridge,
        99.5
    )


    if (
        ridge_scale
        >
        1e-8
    ):

        ridge /= ridge_scale


    ridge = np.clip(
        ridge,
        0,
        1.5
    )


    score = (
        0.75
        *
        ridge
        +
        0.25
        *
        darkness
    )


    score_scale = np.percentile(
        score,
        99.5
    )


    if (
        score_scale
        >
        1e-8
    ):

        score /= score_scale


    return np.clip(
        score,
        0,
        1.5
    )


# ======================================================================================
# 9. FULL-WIDTH MANUAL GUIDE
# ======================================================================================

def make_full_width_guide(
    anchors,
    width
):

    """
    Critical V7 change.

    Manual PCHIP is used between first and last anchors.

    The guide then extends across the COMPLETE crop:

        x = 0 ... width-1

    The small unclicked margins at the left/right are NOT stretched.
    They are simply represented at their true horizontal duration.
    """

    anchors = np.asarray(
        anchors,
        dtype=np.float32
    )


    xs_full = np.arange(
        width,
        dtype=np.float32
    )


    first_x = float(
        anchors[
            0,
            0
        ]
    )


    last_x = float(
        anchors[
            -1,
            0
        ]
    )


    first_y = float(
        anchors[
            0,
            1
        ]
    )


    last_y = float(
        anchors[
            -1,
            1
        ]
    )


    interpolator = PchipInterpolator(
        anchors[
            :,
            0
        ],
        anchors[
            :,
            1
        ],
        extrapolate=False
    )


    guide = np.empty(
        width,
        dtype=np.float32
    )


    inside = (
        (
            xs_full
            >=
            first_x
        )
        &
        (
            xs_full
            <=
            last_x
        )
    )


    guide[
        inside
    ] = interpolator(
        xs_full[
            inside
        ]
    ).astype(
        np.float32
    )


    # ----------------------------------------------------------------------------------
    # Preserve real horizontal duration outside click span.
    #
    # Since annotation was required to begin <=10% and end >=90%,
    # these margins are short.
    #
    # We use the first/last observed trace level as the conservative guide.
    # Local image snapping can still follow nearby ECG ink.
    # ----------------------------------------------------------------------------------

    guide[
        xs_full
        <
        first_x
    ] = first_y


    guide[
        xs_full
        >
        last_x
    ] = last_y


    return (
        xs_full,
        guide
    )


# ======================================================================================
# 10. FULL-WIDTH LOCAL IMAGE RECONSTRUCTION
# ======================================================================================

def reconstruct_full_width(
    crop,
    anchors
):

    H, W = crop.shape


    anchors = np.asarray(
        anchors,
        dtype=np.float32
    )


    anchors[
        :,
        0
    ] = np.clip(
        anchors[
            :,
            0
        ],
        0,
        W - 1
    )


    anchors[
        :,
        1
    ] = np.clip(
        anchors[
            :,
            1
        ],
        0,
        H - 1
    )


    first_fraction = (
        anchors[
            0,
            0
        ]
        /
        max(
            W - 1,
            1
        )
    )


    last_fraction = (
        anchors[
            -1,
            0
        ]
        /
        max(
            W - 1,
            1
        )
    )


    span_fraction = (
        anchors[
            -1,
            0
        ]
        -
        anchors[
            0,
            0
        ]
    ) / max(
        W - 1,
        1
    )


    # ----------------------------------------------------------------------------------
    # Strict independent horizontal-coverage check.
    # ----------------------------------------------------------------------------------

    if (
        first_fraction
        >
        0.11
    ):

        raise RuntimeError(
            f"First anchor too far from left edge: {first_fraction:.3f}"
        )


    if (
        last_fraction
        <
        0.89
    ):

        raise RuntimeError(
            f"Last anchor too far from right edge: {last_fraction:.3f}"
        )


    if (
        span_fraction
        <
        0.78
    ):

        raise RuntimeError(
            f"Insufficient horizontal coverage: {span_fraction:.3f}"
        )


    # ----------------------------------------------------------------------------------
    # Manual full-width guide
    # ----------------------------------------------------------------------------------

    x_full, guide = make_full_width_guide(
        anchors,
        W
    )


    guide = np.clip(
        guide,
        0,
        H - 1
    )


    score = ecg_score_map(
        crop
    )


    # ----------------------------------------------------------------------------------
    # Search radius is intentionally tiny:
    # manual annotation owns the morphology.
    # ----------------------------------------------------------------------------------

    snap_radius = max(
        3,
        int(
            round(
                0.025
                *
                H
            )
        )
    )


    snap_radius = min(
        snap_radius,
        6
    )


    path = np.zeros(
        W,
        dtype=np.float32
    )


    first_anchor_x = float(
        anchors[
            0,
            0
        ]
    )


    last_anchor_x = float(
        anchors[
            -1,
            0
        ]
    )


    # ----------------------------------------------------------------------------------
    # Independent per-column local snap.
    #
    # No global optimizer can jump to another ECG row.
    # ----------------------------------------------------------------------------------

    previous_y = None


    for x in range(
        W
    ):

        gy = float(
            guide[
                x
            ]
        )


        centre = int(
            round(
                gy
            )
        )


        # Even tighter outside manually annotated span.
        if (
            x
            <
            first_anchor_x
            or
            x
            >
            last_anchor_x
        ):

            local_radius = min(
                3,
                snap_radius
            )

            guide_penalty_weight = 0.35

        else:

            local_radius = snap_radius

            guide_penalty_weight = 0.22


        lo = max(
            0,
            centre
            -
            local_radius
        )


        hi = min(
            H - 1,
            centre
            +
            local_radius
        )


        ys = np.arange(
            lo,
            hi + 1,
            dtype=int
        )


        distance_to_guide = np.abs(
            ys
            -
            gy
        )


        objective = (

            2.0
            *
            score[
                ys,
                x
            ]

            -

            guide_penalty_weight
            *
            distance_to_guide
        )


        # --------------------------------------------------------------------------------
        # Tiny continuity preference.
        #
        # It is deliberately weak enough not to suppress steep QRS transitions.
        # --------------------------------------------------------------------------------

        if (
            previous_y
            is not None
        ):

            objective -= (
                0.015
                *
                np.abs(
                    ys
                    -
                    previous_y
                )
            )


        best_y = ys[
            np.argmax(
                objective
            )
        ]


        path[
            x
        ] = float(
            best_y
        )


        previous_y = float(
            best_y
        )


    # ----------------------------------------------------------------------------------
    # Force exact/near-exact agreement at manually clicked x positions.
    # ----------------------------------------------------------------------------------

    for ax, ay in anchors:

        xi = int(
            round(
                ax
            )
        )


        xi = int(
            np.clip(
                xi,
                0,
                W - 1
            )
        )


        path[
            xi
        ] = float(
            ay
        )


    # ----------------------------------------------------------------------------------
    # ECG voltage orientation:
    #
    # image y increases downward,
    # voltage increases upward.
    # ----------------------------------------------------------------------------------

    signal = (
        -path
    )


    signal -= np.median(
        signal
    )


    # ----------------------------------------------------------------------------------
    # Entire crop width -> 1000 points.
    #
    # This now has exactly the same interpretation for every targeted lead.
    # ----------------------------------------------------------------------------------

    old_t = np.linspace(
        0,
        1,
        W
    )


    new_t = np.linspace(
        0,
        1,
        1000
    )


    signal_1000 = np.interp(
        new_t,
        old_t,
        signal
    ).astype(
        np.float32
    )


    signal_1000 -= np.mean(
        signal_1000
    )


    sd = np.std(
        signal_1000
    )


    if (
        sd
        >
        1e-8
    ):

        signal_1000 /= sd


    # ==================================================================================
    # QC METRICS
    # ==================================================================================

    deviation = np.abs(
        path
        -
        guide
    )


    y_idx = np.clip(
        np.round(
            path
        ).astype(
            int
        ),
        0,
        H - 1
    )


    path_scores = score[
        y_idx,
        np.arange(
            W
        )
    ]


    # ----------------------------------------------------------------------------------
    # The key clipping test.
    # ----------------------------------------------------------------------------------

    edge_margin = max(
        3,
        int(
            round(
                0.02
                *
                H
            )
        )
    )


    top_edge_contact = (
        path
        <=
        edge_margin
    )


    bottom_edge_contact = (
        path
        >=
        (
            H
            -
            1
            -
            edge_margin
        )
    )


    edge_contact = (
        top_edge_contact
        |
        bottom_edge_contact
    )


    metrics = {

        "n_anchors":
            int(
                len(
                    anchors
                )
            ),

        "crop_width":
            int(
                W
            ),

        "crop_height":
            int(
                H
            ),

        "first_anchor_fraction":
            float(
                first_fraction
            ),

        "last_anchor_fraction":
            float(
                last_fraction
            ),

        "annotation_span_fraction":
            float(
                span_fraction
            ),

        "reconstruction_span_fraction":
            1.0,

        "snap_radius_px":
            int(
                snap_radius
            ),

        "median_snap_distance":
            float(
                np.median(
                    deviation
                )
            ),

        "p95_snap_distance":
            float(
                np.percentile(
                    deviation,
                    95
                )
            ),

        "max_snap_distance":
            float(
                np.max(
                    deviation
                )
            ),

        "mean_path_score":
            float(
                np.mean(
                    path_scores
                )
            ),

        "median_path_score":
            float(
                np.median(
                    path_scores
                )
            ),

        "top_edge_contact_fraction":
            float(
                np.mean(
                    top_edge_contact
                )
            ),

        "bottom_edge_contact_fraction":
            float(
                np.mean(
                    bottom_edge_contact
                )
            ),

        "any_edge_contact_fraction":
            float(
                np.mean(
                    edge_contact
                )
            ),

        "minimum_distance_to_crop_edge_px":
            float(
                np.min(
                    np.minimum(
                        path,
                        (
                            H
                            -
                            1
                            -
                            path
                        )
                    )
                )
            ),
    }


    return {

        "x":
            np.arange(
                W,
                dtype=np.float32
            ),

        "y":
            path.astype(
                np.float32
            ),

        "guide":
            guide.astype(
                np.float32
            ),

        "anchors":
            anchors.astype(
                np.float32
            ),

        "signal":
            signal_1000.astype(
                np.float32
            ),

        "metrics":
            metrics,
    }


# ======================================================================================
# 11. PATCH ONLY THE 14 TARGET LEADS
# ======================================================================================

results = {}

log_rows = []

metric_rows = []


print(
    "\n" +
    "=" * 100
)

print(
    "PATCHING TARGETED LEADS"
)

print(
    "=" * 100
)


for ecg, leads in TARGET_LEADS.items():

    for lead in leads:

        key = (
            ecg
            +
            "::"
            +
            lead
        )


        ann = annotations[
            key
        ]


        box = [
            int(
                round(
                    value
                )
            )

            for value
            in ann[
                "original_box"
            ]
        ]


        x0, y0, x1, y1 = (
            box
        )


        source_path = resolve_source_image(
            geometry[
                ecg
            ][
                "image_path"
            ]
        )


        source_gray = load_gray(
            source_path
        )


        crop = source_gray[
            y0:y1,
            x0:x1
        ]


        anchors = browser_points_to_crop(
            ann
        )


        result = reconstruct_full_width(
            crop=crop,
            anchors=anchors
        )


        ecg_idx = ECG_ORDER.index(
            ecg
        )


        lead_idx = LEAD_ORDER.index(
            lead
        )


        # --------------------------------------------------------------------------------
        # Replace ONLY this one lead.
        # --------------------------------------------------------------------------------

        X_final[
            ecg_idx,
            lead_idx,
            :,
            0
        ] = result[
            "signal"
        ]


        results[
            (
                ecg,
                lead
            )
        ] = {

            "crop":
                crop,

            "box":
                box,

            **result
        }


        log_rows.append({

            "ECG":
                ecg,

            "lead":
                lead,

            "status":
                "targeted_expanded_crop_patch",

            "n_anchors":
                len(
                    anchors
                ),
        })


        metric_rows.append({

            "ECG":
                ecg,

            "lead":
                lead,

            **result[
                "metrics"
            ],
        })


        print(
            f"{ecg:22s} "
            f"{lead:4s} "
            f"-> patched using "
            f"{len(anchors):2d} anchors"
        )


print(
    "\n✓ All 14 targeted leads patched."
)


# ======================================================================================
# 12. CONFIRM THAT NOTHING ELSE CHANGED
# ======================================================================================

changed_leads = []


for ei, ecg in enumerate(
    ECG_ORDER
):

    for li, lead in enumerate(
        LEAD_ORDER
    ):

        max_diff = float(
            np.max(
                np.abs(
                    X_final[
                        ei,
                        li,
                        :,
                        0
                    ]
                    -
                    X_v6[
                        ei,
                        li,
                        :,
                        0
                    ]
                )
            )
        )


        if (
            max_diff
            >
            1e-7
        ):

            changed_leads.append(
                (
                    ecg,
                    lead,
                    max_diff
                )
            )


actual_changed_set = set(
    (
        ecg,
        lead
    )

    for ecg, lead, _
    in changed_leads
)


expected_changed_set = set()


for ecg, leads in TARGET_LEADS.items():

    for lead in leads:

        expected_changed_set.add(
            (
                ecg,
                lead
            )
        )


unexpected_changes = (
    actual_changed_set
    -
    expected_changed_set
)


if len(
    unexpected_changes
) > 0:

    raise RuntimeError(
        "\nUnexpected non-target leads changed:\n"
        +
        "\n".join(
            str(x)
            for x
            in sorted(
                unexpected_changes
            )
        )
    )


print(
    "\nChanged lead count:",
    len(
        actual_changed_set
    )
)


print(
    "Expected targeted lead count:",
    len(
        expected_changed_set
    )
)


print(
    "\n✓ No non-target V6 lead was modified."
)


# ======================================================================================
# 13. SAVE PER-LEAD PATH DATA
# ======================================================================================

for (
    ecg,
    lead
), result in results.items():

    np.savez_compressed(

        OUT_DIR
        /
        f"PATH_{ecg}_{lead}.npz",

        x=result[
            "x"
        ],

        y=result[
            "y"
        ],

        guide=result[
            "guide"
        ],

        anchors=result[
            "anchors"
        ],

        signal=result[
            "signal"
        ],
    )


# ======================================================================================
# 14. TARGETED SOURCE-OVERLAY QC
# ======================================================================================

print(
    "\nGenerating FINAL_OVERLAY figures..."
)


for ecg, leads in TARGET_LEADS.items():

    n = len(
        leads
    )


    fig, axes = plt.subplots(

        nrows=n,

        ncols=2,

        figsize=(
            17,
            max(
                4,
                3.4
                *
                n
            )
        ),

        squeeze=False
    )


    for row_idx, lead in enumerate(
        leads
    ):

        result = results[
            (
                ecg,
                lead
            )
        ]


        crop = result[
            "crop"
        ]


        x = result[
            "x"
        ]


        y = result[
            "y"
        ]


        guide = result[
            "guide"
        ]


        anchors = result[
            "anchors"
        ]


        # ==================================================================================
        # LEFT — SOURCE + GUIDE + FINAL PATH
        # ==================================================================================

        ax = axes[
            row_idx,
            0
        ]


        ax.imshow(
            crop,
            cmap="gray",
            origin="upper",
            aspect="auto"
        )


        ax.plot(
            x,
            guide,
            linestyle="--",
            linewidth=1.0,
            alpha=0.8,
            label="manual full-width guide"
        )


        ax.plot(
            x,
            y,
            color="red",
            linewidth=1.6,
            label="final path"
        )


        ax.scatter(
            anchors[
                :,
                0
            ],
            anchors[
                :,
                1
            ],
            s=20,
            facecolors="none",
            edgecolors="yellow",
            linewidths=1.0,
            label="manual anchors"
        )


        ax.set_xlim(
            0,
            crop.shape[
                1
            ]
            -
            1
        )


        ax.set_ylim(
            crop.shape[
                0
            ]
            -
            1,
            0
        )


        ax.set_title(
            f"{lead}: expanded source + FINAL path"
        )


        ax.set_xticks([])
        ax.set_yticks([])


        if (
            row_idx
            ==
            0
        ):

            ax.legend(
                fontsize=7,
                loc="upper right"
            )


        # ==================================================================================
        # RIGHT — FINAL PATCHED PSEUDO-WAVEFORM
        # ==================================================================================

        ax = axes[
            row_idx,
            1
        ]


        ei = ECG_ORDER.index(
            ecg
        )


        li = LEAD_ORDER.index(
            lead
        )


        signal = X_final[
            ei,
            li,
            :,
            0
        ]


        ax.plot(
            signal,
            linewidth=1.0
        )


        ax.axhline(
            0,
            linewidth=0.45,
            alpha=0.35
        )


        ax.set_xlim(
            0,
            999
        )


        ax.set_title(
            f"{lead}: final full-duration pseudo-waveform"
        )


        ax.set_xticks([])
        ax.set_yticks([])


    fig.suptitle(
        f"{ecg} — FINAL TARGETED EXPANDED-CROP QC",
        fontsize=18,
        y=0.995
    )


    fig.tight_layout()


    fig.savefig(

        OUT_DIR
        /
        f"FINAL_OVERLAY_{ecg}.png",

        dpi=200,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ======================================================================================
# 15. ALL TEN FINAL 12-LEAD SIGNAL SHEETS
# ======================================================================================

print(
    "Generating FINAL_SIGNAL figures..."
)


for ei, ecg in enumerate(
    ECG_ORDER
):

    fig, axes = plt.subplots(

        3,
        4,

        figsize=(
            18,
            10
        ),

        constrained_layout=True
    )


    for r in range(
        3
    ):

        for c in range(
            4
        ):

            lead = DISPLAY_GRID[
                r
            ][
                c
            ]


            li = LEAD_ORDER.index(
                lead
            )


            signal = X_final[
                ei,
                li,
                :,
                0
            ]


            ax = axes[
                r,
                c
            ]


            ax.plot(
                signal,
                linewidth=1.0
            )


            ax.axhline(
                0,
                linewidth=0.4,
                alpha=0.3
            )


            if (
                ecg,
                lead
            ) in expected_changed_set:

                suffix = (
                    " [V7 TARGETED]"
                )

            else:

                suffix = (
                    " [V6 RETAINED]"
                )


            ax.set_title(
                lead
                +
                suffix
            )


            ax.set_xlim(
                0,
                999
            )


            ax.set_xticks([])
            ax.set_yticks([])


    fig.suptitle(
        f"{ecg} — FINAL V7 candidate 12-lead pseudo-waveform",
        fontsize=18
    )


    fig.savefig(

        OUT_DIR
        /
        f"FINAL_SIGNAL_{ecg}.png",

        dpi=200,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ======================================================================================
# 16. TARGETED QC METRICS
# ======================================================================================

log_df = pd.DataFrame(
    log_rows
)


metrics_df = pd.DataFrame(
    metric_rows
)


# --------------------------------------------------------------------------------------
# Flag remaining possible crop-edge contact.
# --------------------------------------------------------------------------------------

metrics_df[
    "edge_contact_flag"
] = (
    metrics_df[
        "any_edge_contact_fraction"
    ]
    >
    0.01
)


metrics_df[
    "very_close_to_edge_flag"
] = (
    metrics_df[
        "minimum_distance_to_crop_edge_px"
    ]
    <
    3
)


metrics_df[
    "automatic_QC_flag"
] = (
    metrics_df[
        "edge_contact_flag"
    ]
    |
    metrics_df[
        "very_close_to_edge_flag"
    ]
)


log_df.to_csv(

    OUT_DIR
    /
    "targeted_patch_log.csv",

    index=False
)


metrics_df.to_csv(

    OUT_DIR
    /
    "targeted_patch_metrics.csv",

    index=False
)


print(
    "\n" +
    "=" * 100
)

print(
    "TARGETED PATCH METRICS"
)

print(
    "=" * 100
)


print(
    metrics_df[
        [
            "ECG",
            "lead",
            "n_anchors",
            "first_anchor_fraction",
            "last_anchor_fraction",
            "annotation_span_fraction",
            "reconstruction_span_fraction",
            "median_snap_distance",
            "p95_snap_distance",
            "minimum_distance_to_crop_edge_px",
            "any_edge_contact_fraction",
            "automatic_QC_flag",
        ]
    ]
    .to_string(
        index=False
    )
)


flagged = metrics_df[
    metrics_df[
        "automatic_QC_flag"
    ]
]


print(
    "\nAUTOMATIC EDGE FLAGS"
)


if len(
    flagged
) == 0:

    print(
        "None."
    )

else:

    print(
        flagged[
            [
                "ECG",
                "lead",
                "minimum_distance_to_crop_edge_px",
                "top_edge_contact_fraction",
                "bottom_edge_contact_fraction",
                "any_edge_contact_fraction",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ======================================================================================
# 17. WHOLE FINAL ARRAY NUMERICAL QC
# ======================================================================================

qc_rows = []


for ei, ecg in enumerate(
    ECG_ORDER
):

    for li, lead in enumerate(
        LEAD_ORDER
    ):

        signal = X_final[
            ei,
            li,
            :,
            0
        ]


        qc_rows.append({

            "ECG":
                ecg,

            "lead":
                lead,

            "finite":
                bool(
                    np.all(
                        np.isfinite(
                            signal
                        )
                    )
                ),

            "mean":
                float(
                    np.mean(
                        signal
                    )
                ),

            "std":
                float(
                    np.std(
                        signal
                    )
                ),

            "min":
                float(
                    np.min(
                        signal
                    )
                ),

            "max":
                float(
                    np.max(
                        signal
                    )
                ),

            "median_abs_diff":
                float(
                    np.median(
                        np.abs(
                            np.diff(
                                signal
                            )
                        )
                    )
                ),

            "max_abs_diff":
                float(
                    np.max(
                        np.abs(
                            np.diff(
                                signal
                            )
                        )
                    )
                ),
        })


qc_df = pd.DataFrame(
    qc_rows
)


if not qc_df[
    "finite"
].all():

    raise RuntimeError(
        "Non-finite values detected."
    )


qc_df.to_csv(

    OUT_DIR
    /
    "final_array_qc.csv",

    index=False
)


# ======================================================================================
# 18. SAVE FINAL ARRAY
# ======================================================================================

FINAL_ARRAY_PATH = (
    OUT_DIR
    /
    "X_gaze_final.npy"
)


np.save(
    FINAL_ARRAY_PATH,
    X_final
)


assert (
    X_final.shape
    ==
    (
        10,
        12,
        1000,
        1
    )
)


# ======================================================================================
# 19. HASHES
# ======================================================================================

def sha256_file(
    path
):

    h = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024
                *
                1024
            )


            if not block:

                break


            h.update(
                block
            )


    return h.hexdigest()


V6_HASH = sha256_file(
    V6_ARRAY_PATH
)


V7_HASH = sha256_file(
    FINAL_ARRAY_PATH
)


# ======================================================================================
# 20. METADATA
# ======================================================================================

metadata = {

    "created":
        datetime.now().isoformat(),

    "parent_dataset":
        str(
            V6_ARRAY_PATH
        ),

    "parent_dataset_sha256":
        V6_HASH,

    "targeted_annotation_json":
        str(
            TARGETED_JSON_PATH
        ),

    "manual_geometry":
        str(
            GEOMETRY_PATH
        ),

    "final_dataset":
        str(
            FINAL_ARRAY_PATH
        ),

    "final_dataset_sha256":
        V7_HASH,

    "shape":
        list(
            X_final.shape
        ),

    "ecg_order":
        ECG_ORDER,

    "lead_order":
        LEAD_ORDER,

    "number_targeted_leads_replaced":
        len(
            expected_changed_set
        ),

    "targeted_leads":
        TARGET_LEADS,

    "method_summary": (
        "The accepted V6 pseudo-waveform dataset was used as the parent dataset. "
        "Only 14 leads showing residual crop-boundary clipping underwent targeted "
        "replacement. Each was re-annotated using an expanded source-image crop. "
        "Manual anchor points defined a shape-preserving PCHIP waveform trajectory. "
        "The trajectory was reconstructed across the complete horizontal diagnostic "
        "lead interval, including the small margins outside the first and last manual "
        "anchors, thereby preserving full segment duration. Pixel-level refinement "
        "was restricted to local vertical image snapping of at most six pixels. "
        "All non-target V6 leads were retained exactly."
    ),

    "model_training_performed":
        False,
}


with open(

    OUT_DIR
    /
    "metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )


# ======================================================================================
# 21. FINAL REPORT
# ======================================================================================

print(
    "\n\n" +
    "=" * 100
)

print(
    "V7 TARGETED PATCH COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nFinal candidate dataset:"
)

print(
    "   ",
    FINAL_ARRAY_PATH
)


print(
    "\nShape:"
)

print(
    "   ",
    X_final.shape
)


print(
    "\nParent V6 SHA256:"
)

print(
    "   ",
    V6_HASH
)


print(
    "\nFinal V7 SHA256:"
)

print(
    "   ",
    V7_HASH
)


print(
    """
FINAL QC TO INSPECT:

    final_digitisation_v7/FINAL_OVERLAY_Hyperkalemia.png
    final_digitisation_v7/FINAL_OVERLAY_VTach.png
    final_digitisation_v7/FINAL_OVERLAY_VentPaced.png
    final_digitisation_v7/FINAL_OVERLAY_LBBB.png
    final_digitisation_v7/FINAL_OVERLAY_CompleteHeartBlock.png


On each overlay:

    yellow circles = your targeted manual anchors
    dashed line    = full-width manual guide
    red line       = final locally snapped path


We specifically need to confirm:

    • VT peaks/troughs are no longer cropped
    • paced V2–V5 are no longer clipped
    • LBBB V1–V3 have no flat crop-boundary troughs
    • complete-heart-block V1/V2 peaks are complete
    • hyperkalaemia V3 is complete
    • red path remains on the intended lead despite the taller crop


Also paste the printed:

    AUTOMATIC EDGE FLAGS

output.

DO NOT TRAIN THE MODEL YET.
"""
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "final_digitisation_v7/targeted_patch_metrics.csv"
)

flagged = df[
    df["automatic_QC_flag"] == True
]

print("AUTOMATIC EDGE FLAGS")
print("=" * 80)

if len(flagged) == 0:
    print("None.")
else:
    print(
        flagged[
            [
                "ECG",
                "lead",
                "minimum_distance_to_crop_edge_px",
                "top_edge_contact_fraction",
                "bottom_edge_contact_fraction",
                "any_edge_contact_fraction",
            ]
        ].to_string(index=False)
    )

In [ ]:
# ======================================================================================
# FINAL DIGITISATION PATCH — V8
#
# Parent:
#     final_digitisation_v7/X_gaze_final.npy
#
# New annotations:
#     LOCAL_USER_HOME/Downloads/ecg_anchor_annotations_LBBB_final*.json
#
# Replaces ONLY:
#     LBBB V1
#     LBBB V2
#     LBBB V3
#
# Output:
#     final_digitisation_v8/
#         X_gaze_final.npy
#         FINAL_OVERLAY_LBBB.png
#         FINAL_SIGNAL_LBBB.png
#         lbbb_final_metrics.csv
#         metadata.json
#
# DO NOT TRAIN YET.
# ======================================================================================

import json
import hashlib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from scipy.ndimage import gaussian_filter
from scipy.interpolate import PchipInterpolator

warnings.filterwarnings("ignore")


# ======================================================================================
# 1. PATHS
# ======================================================================================

OUT_DIR = Path("final_digitisation_v8")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PARENT_PATH = Path(
    "final_digitisation_v7/X_gaze_final.npy"
)

GEOMETRY_PATH = Path(
    "manual_digitization_qc/manual_geometry.json"
)

DOWNLOADS = Path(
    "LOCAL_USER_HOME/Downloads"
)

json_candidates = list(
    DOWNLOADS.glob(
        "ecg_anchor_annotations_LBBB_final*.json"
    )
)

if not json_candidates:
    raise FileNotFoundError(
        "Could not find ecg_anchor_annotations_LBBB_final*.json "
        "in LOCAL_USER_HOME/Downloads/"
    )

ANNOTATION_PATH = max(
    json_candidates,
    key=lambda p: p.stat().st_mtime
)

for p in [
    PARENT_PATH,
    GEOMETRY_PATH,
    ANNOTATION_PATH
]:
    if not p.exists():
        raise FileNotFoundError(p)

print("=" * 100)
print("INPUTS")
print("=" * 100)
print("Parent V7:     ", PARENT_PATH)
print("Geometry:      ", GEOMETRY_PATH)
print("LBBB JSON:     ", ANNOTATION_PATH)


# ======================================================================================
# 2. LOAD
# ======================================================================================

X_v7 = np.load(
    PARENT_PATH
).astype(np.float32)

if X_v7.shape != (10, 12, 1000, 1):
    raise ValueError(
        f"Unexpected parent shape: {X_v7.shape}"
    )

X_final = X_v7.copy()

with open(
    GEOMETRY_PATH,
    "r"
) as f:
    geometry = json.load(f)

with open(
    ANNOTATION_PATH,
    "r"
) as f:
    annotation_json = json.load(f)

annotations = annotation_json.get(
    "annotations",
    {}
)

if not annotations:
    raise RuntimeError(
        "No annotations found in final LBBB JSON."
    )


# ======================================================================================
# 3. ORDERS
# ======================================================================================

ECG_ORDER = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]

LEAD_ORDER = [
    "I",
    "II",
    "III",
    "aVR",
    "aVL",
    "aVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]

TARGETS = [
    ("LBBB", "V1"),
    ("LBBB", "V2"),
    ("LBBB", "V3"),
]

EXPECTED_KEYS = {
    f"{ecg}::{lead}"
    for ecg, lead in TARGETS
}


# ======================================================================================
# 4. VERIFY JSON
# ======================================================================================

print("\n" + "=" * 100)
print("ANNOTATION CHECK")
print("=" * 100)

for key in sorted(EXPECTED_KEYS):

    if key not in annotations:
        raise RuntimeError(
            f"Missing annotation: {key}"
        )

    ann = annotations[key]

    n = len(
        ann.get(
            "points",
            []
        )
    )

    status = ann.get(
        "status"
    )

    print(
        f"{key:12s} | "
        f"status={status:10s} | "
        f"anchors={n}"
    )

    if status != "corrected":
        raise RuntimeError(
            f"{key} is not marked corrected."
        )

    if n < 14:
        raise RuntimeError(
            f"{key} has only {n} anchors."
        )

print("\n✓ All three LBBB annotations are complete.")


# ======================================================================================
# 5. SOURCE IMAGE HELPERS
# ======================================================================================

def resolve_source_image(stored_path):

    p = Path(stored_path)

    if p.exists():
        return p

    matches = list(
        Path(".").glob(
            f"**/{p.name}"
        )
    )

    if not matches:
        raise FileNotFoundError(
            f"Could not locate {p.name}"
        )

    return matches[0]


def load_gray(path):

    return (
        np.asarray(
            Image.open(path).convert("L")
        )
        .astype(np.float32)
        / 255.0
    )


# ======================================================================================
# 6. BROWSER -> ORIGINAL CROP COORDINATES
# ======================================================================================

def browser_points_to_crop(ann):

    points = np.asarray(
        [
            [
                float(p["x"]),
                float(p["y"])
            ]
            for p in ann["points"]
        ],
        dtype=np.float32
    )

    points[:, 0] *= (
        float(
            ann["original_crop_width"]
        )
        /
        float(
            ann["display_width"]
        )
    )

    points[:, 1] *= (
        float(
            ann["original_crop_height"]
        )
        /
        float(
            ann["display_height"]
        )
    )

    points = points[
        np.argsort(
            points[:, 0]
        )
    ]

    cleaned = [
        points[0]
    ]

    for p in points[1:]:

        if (
            p[0]
            -
            cleaned[-1][0]
            >=
            0.5
        ):
            cleaned.append(p)

    points = np.asarray(
        cleaned,
        dtype=np.float32
    )

    if len(points) < 12:
        raise RuntimeError(
            "Too few unique anchors after coordinate conversion."
        )

    return points


# ======================================================================================
# 7. ECG IMAGE SCORE
# ======================================================================================

def ecg_score_map(gray):

    gray = np.asarray(
        gray,
        dtype=np.float32
    )

    p1 = np.percentile(
        gray,
        1
    )

    p99 = np.percentile(
        gray,
        99
    )

    if p99 > p1 + 1e-8:
        norm = (
            gray - p1
        ) / (
            p99 - p1
        )
    else:
        norm = gray.copy()

    norm = np.clip(
        norm,
        0,
        1
    )

    darkness = 1.0 - norm

    fine = gaussian_filter(
        norm,
        sigma=0.45
    )

    background = gaussian_filter(
        norm,
        sigma=2.2
    )

    ridge = np.clip(
        background - fine,
        0,
        None
    )

    scale = np.percentile(
        ridge,
        99.5
    )

    if scale > 1e-8:
        ridge /= scale

    ridge = np.clip(
        ridge,
        0,
        1.5
    )

    score = (
        0.75 * ridge
        +
        0.25 * darkness
    )

    scale = np.percentile(
        score,
        99.5
    )

    if scale > 1e-8:
        score /= scale

    return np.clip(
        score,
        0,
        1.5
    )


# ======================================================================================
# 8. FULL-WIDTH MANUAL GUIDE
# ======================================================================================

def full_width_guide(
    anchors,
    width
):

    xs = np.arange(
        width,
        dtype=np.float32
    )

    first_x = float(
        anchors[0, 0]
    )

    last_x = float(
        anchors[-1, 0]
    )

    first_y = float(
        anchors[0, 1]
    )

    last_y = float(
        anchors[-1, 1]
    )

    interpolator = PchipInterpolator(
        anchors[:, 0],
        anchors[:, 1],
        extrapolate=False
    )

    guide = np.empty(
        width,
        dtype=np.float32
    )

    inside = (
        (xs >= first_x)
        &
        (xs <= last_x)
    )

    guide[inside] = interpolator(
        xs[inside]
    ).astype(
        np.float32
    )

    # True horizontal duration retained.
    guide[xs < first_x] = first_y
    guide[xs > last_x] = last_y

    return xs, guide


# ======================================================================================
# 9. RECONSTRUCTION
# ======================================================================================

def reconstruct(
    crop,
    anchors
):

    H, W = crop.shape

    anchors = np.asarray(
        anchors,
        dtype=np.float32
    )

    anchors[:, 0] = np.clip(
        anchors[:, 0],
        0,
        W - 1
    )

    anchors[:, 1] = np.clip(
        anchors[:, 1],
        0,
        H - 1
    )

    first_fraction = (
        anchors[0, 0]
        /
        max(W - 1, 1)
    )

    last_fraction = (
        anchors[-1, 0]
        /
        max(W - 1, 1)
    )

    span_fraction = (
        anchors[-1, 0]
        -
        anchors[0, 0]
    ) / max(
        W - 1,
        1
    )

    if first_fraction > 0.11:
        raise RuntimeError(
            f"First anchor too far right: {first_fraction:.3f}"
        )

    if last_fraction < 0.89:
        raise RuntimeError(
            f"Last anchor too far left: {last_fraction:.3f}"
        )

    if span_fraction < 0.78:
        raise RuntimeError(
            f"Insufficient coverage: {span_fraction:.3f}"
        )

    xs, guide = full_width_guide(
        anchors,
        W
    )

    guide = np.clip(
        guide,
        0,
        H - 1
    )

    score = ecg_score_map(
        crop
    )

    # Manual trajectory dominates.
    snap_radius = min(
        6,
        max(
            3,
            int(
                round(
                    0.025 * H
                )
            )
        )
    )

    path = np.zeros(
        W,
        dtype=np.float32
    )

    first_x = anchors[0, 0]
    last_x = anchors[-1, 0]

    previous_y = None

    for x in range(W):

        gy = float(
            guide[x]
        )

        centre = int(
            round(gy)
        )

        if (
            x < first_x
            or
            x > last_x
        ):
            radius = min(
                3,
                snap_radius
            )

            distance_weight = 0.35

        else:
            radius = snap_radius
            distance_weight = 0.22

        lo = max(
            0,
            centre - radius
        )

        hi = min(
            H - 1,
            centre + radius
        )

        ys = np.arange(
            lo,
            hi + 1
        )

        objective = (
            2.0
            *
            score[
                ys,
                x
            ]
            -
            distance_weight
            *
            np.abs(
                ys - gy
            )
        )

        if previous_y is not None:
            objective -= (
                0.015
                *
                np.abs(
                    ys - previous_y
                )
            )

        best_y = ys[
            np.argmax(
                objective
            )
        ]

        path[x] = float(
            best_y
        )

        previous_y = float(
            best_y
        )

    # Force manual anchor locations.
    for ax, ay in anchors:

        xi = int(
            np.clip(
                round(ax),
                0,
                W - 1
            )
        )

        path[xi] = float(
            ay
        )

    # ------------------------------------------------------------------
    # Image y -> ECG voltage.
    # ------------------------------------------------------------------

    signal = -path

    signal -= np.median(
        signal
    )

    signal_1000 = np.interp(
        np.linspace(
            0,
            1,
            1000
        ),
        np.linspace(
            0,
            1,
            W
        ),
        signal
    ).astype(
        np.float32
    )

    signal_1000 -= np.mean(
        signal_1000
    )

    sd = np.std(
        signal_1000
    )

    if sd > 1e-8:
        signal_1000 /= sd

    # ------------------------------------------------------------------
    # FINAL EDGE QC
    # ------------------------------------------------------------------

    distance_top = path

    distance_bottom = (
        H - 1 - path
    )

    distance_edge = np.minimum(
        distance_top,
        distance_bottom
    )

    edge_margin = max(
        3,
        int(
            round(
                0.02 * H
            )
        )
    )

    top_contact = (
        path
        <=
        edge_margin
    )

    bottom_contact = (
        path
        >=
        H - 1 - edge_margin
    )

    edge_contact = (
        top_contact
        |
        bottom_contact
    )

    # Longest continuous run of edge contact.
    longest_run = 0
    current_run = 0

    for value in edge_contact:

        if value:
            current_run += 1
            longest_run = max(
                longest_run,
                current_run
            )
        else:
            current_run = 0

    deviation = np.abs(
        path - guide
    )

    y_indices = np.clip(
        np.round(
            path
        ).astype(int),
        0,
        H - 1
    )

    path_score = score[
        y_indices,
        np.arange(W)
    ]

    metrics = {

        "n_anchors":
            len(anchors),

        "crop_width":
            W,

        "crop_height":
            H,

        "first_anchor_fraction":
            float(
                first_fraction
            ),

        "last_anchor_fraction":
            float(
                last_fraction
            ),

        "span_fraction":
            float(
                span_fraction
            ),

        "snap_radius_px":
            snap_radius,

        "median_snap_distance_px":
            float(
                np.median(
                    deviation
                )
            ),

        "p95_snap_distance_px":
            float(
                np.percentile(
                    deviation,
                    95
                )
            ),

        "minimum_distance_to_edge_px":
            float(
                np.min(
                    distance_edge
                )
            ),

        "top_edge_contact_fraction":
            float(
                np.mean(
                    top_contact
                )
            ),

        "bottom_edge_contact_fraction":
            float(
                np.mean(
                    bottom_contact
                )
            ),

        "any_edge_contact_fraction":
            float(
                np.mean(
                    edge_contact
                )
            ),

        "longest_edge_contact_run_px":
            int(
                longest_run
            ),

        "mean_path_score":
            float(
                np.mean(
                    path_score
                )
            ),
    }

    return {
        "x":
            np.arange(
                W,
                dtype=np.float32
            ),

        "y":
            path,

        "guide":
            guide,

        "anchors":
            anchors,

        "signal":
            signal_1000,

        "metrics":
            metrics,
    }


# ======================================================================================
# 10. PATCH LBBB V1–V3
# ======================================================================================

results = {}
metric_rows = []

lbbb_index = ECG_ORDER.index(
    "LBBB"
)

source_path = resolve_source_image(
    geometry[
        "LBBB"
    ][
        "image_path"
    ]
)

source_gray = load_gray(
    source_path
)

print("\n" + "=" * 100)
print("FINAL LBBB PATCH")
print("=" * 100)

for lead in [
    "V1",
    "V2",
    "V3"
]:

    key = f"LBBB::{lead}"

    ann = annotations[
        key
    ]

    box = [
        int(
            round(v)
        )
        for v in ann[
            "original_box"
        ]
    ]

    x0, y0, x1, y1 = box

    crop = source_gray[
        y0:y1,
        x0:x1
    ]

    anchors = browser_points_to_crop(
        ann
    )

    result = reconstruct(
        crop,
        anchors
    )

    lead_index = LEAD_ORDER.index(
        lead
    )

    X_final[
        lbbb_index,
        lead_index,
        :,
        0
    ] = result[
        "signal"
    ]

    results[
        lead
    ] = {
        "crop":
            crop,
        **result
    }

    metric_rows.append(
        {
            "ECG":
                "LBBB",
            "lead":
                lead,
            **result[
                "metrics"
            ]
        }
    )

    print(
        f"LBBB {lead}: "
        f"{len(anchors)} anchors | "
        f"crop {crop.shape}"
    )


# ======================================================================================
# 11. VERIFY ONLY THREE LEADS CHANGED FROM V7
# ======================================================================================

changed = []

for ei, ecg in enumerate(
    ECG_ORDER
):

    for li, lead in enumerate(
        LEAD_ORDER
    ):

        max_diff = float(
            np.max(
                np.abs(
                    X_final[
                        ei,
                        li,
                        :,
                        0
                    ]
                    -
                    X_v7[
                        ei,
                        li,
                        :,
                        0
                    ]
                )
            )
        )

        if max_diff > 1e-7:

            changed.append(
                (
                    ecg,
                    lead,
                    max_diff
                )
            )

changed_set = {
    (x[0], x[1])
    for x in changed
}

expected_set = {
    ("LBBB", "V1"),
    ("LBBB", "V2"),
    ("LBBB", "V3"),
}

if changed_set != expected_set:

    raise RuntimeError(
        "\nUnexpected changed leads.\n"
        f"Expected: {expected_set}\n"
        f"Actual:   {changed_set}"
    )

print(
    "\n✓ Exactly LBBB V1/V2/V3 changed; all other V7 leads are untouched."
)


# ======================================================================================
# 12. EDGE QC TABLE
# ======================================================================================

metrics_df = pd.DataFrame(
    metric_rows
)

metrics_df[
    "final_edge_flag"
] = (
    (
        metrics_df[
            "any_edge_contact_fraction"
        ]
        >
        0.01
    )
    |
    (
        metrics_df[
            "minimum_distance_to_edge_px"
        ]
        <
        3
    )
    |
    (
        metrics_df[
            "longest_edge_contact_run_px"
        ]
        >
        5
    )
)

metrics_df.to_csv(
    OUT_DIR
    /
    "lbbb_final_metrics.csv",
    index=False
)

print("\n" + "=" * 100)
print("FINAL LBBB EDGE QC")
print("=" * 100)

print(
    metrics_df[
        [
            "lead",
            "n_anchors",
            "crop_height",
            "minimum_distance_to_edge_px",
            "top_edge_contact_fraction",
            "bottom_edge_contact_fraction",
            "any_edge_contact_fraction",
            "longest_edge_contact_run_px",
            "final_edge_flag",
        ]
    ].to_string(
        index=False
    )
)


# ======================================================================================
# 13. FINAL LBBB OVERLAY
# ======================================================================================

fig, axes = plt.subplots(
    3,
    2,
    figsize=(
        17,
        12
    ),
    constrained_layout=True
)

for row, lead in enumerate(
    [
        "V1",
        "V2",
        "V3"
    ]
):

    result = results[
        lead
    ]

    crop = result[
        "crop"
    ]

    x = result[
        "x"
    ]

    y = result[
        "y"
    ]

    guide = result[
        "guide"
    ]

    anchors = result[
        "anchors"
    ]

    # Source overlay
    ax = axes[
        row,
        0
    ]

    ax.imshow(
        crop,
        cmap="gray",
        origin="upper",
        aspect="auto"
    )

    ax.plot(
        x,
        guide,
        linestyle="--",
        linewidth=1,
        label="manual full-width guide"
    )

    ax.plot(
        x,
        y,
        color="red",
        linewidth=1.6,
        label="final path"
    )

    ax.scatter(
        anchors[:, 0],
        anchors[:, 1],
        s=20,
        facecolors="none",
        edgecolors="yellow",
        linewidths=1,
        label="manual anchors"
    )

    ax.set_xlim(
        0,
        crop.shape[1] - 1
    )

    ax.set_ylim(
        crop.shape[0] - 1,
        0
    )

    ax.set_title(
        f"{lead}: ultra-expanded source + FINAL path"
    )

    ax.set_xticks([])
    ax.set_yticks([])

    if row == 0:
        ax.legend(
            fontsize=7,
            loc="upper right"
        )

    # Pseudo-waveform
    ax = axes[
        row,
        1
    ]

    li = LEAD_ORDER.index(
        lead
    )

    signal = X_final[
        lbbb_index,
        li,
        :,
        0
    ]

    ax.plot(
        signal,
        linewidth=1
    )

    ax.axhline(
        0,
        linewidth=0.4,
        alpha=0.3
    )

    ax.set_xlim(
        0,
        999
    )

    ax.set_title(
        f"{lead}: FINAL full-duration pseudo-waveform"
    )

    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(
    "LBBB — FINAL V8 ULTRA-EXPANDED QC",
    fontsize=18
)

fig.savefig(
    OUT_DIR
    /
    "FINAL_OVERLAY_LBBB.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)


# ======================================================================================
# 14. FINAL LBBB 12-LEAD SIGNAL SHEET
# ======================================================================================

DISPLAY_GRID = [
    ["I",   "aVR", "V1", "V4"],
    ["II",  "aVL", "V2", "V5"],
    ["III", "aVF", "V3", "V6"],
]

fig, axes = plt.subplots(
    3,
    4,
    figsize=(
        18,
        10
    ),
    constrained_layout=True
)

for r in range(3):

    for c in range(4):

        lead = DISPLAY_GRID[
            r
        ][
            c
        ]

        li = LEAD_ORDER.index(
            lead
        )

        signal = X_final[
            lbbb_index,
            li,
            :,
            0
        ]

        ax = axes[
            r,
            c
        ]

        ax.plot(
            signal,
            linewidth=1
        )

        ax.axhline(
            0,
            linewidth=0.4,
            alpha=0.3
        )

        if lead in [
            "V1",
            "V2",
            "V3"
        ]:
            suffix = " [V8 FINAL]"
        else:
            suffix = " [RETAINED]"

        ax.set_title(
            lead
            +
            suffix
        )

        ax.set_xlim(
            0,
            999
        )

        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle(
    "LBBB — FINAL V8 12-lead candidate",
    fontsize=18
)

fig.savefig(
    OUT_DIR
    /
    "FINAL_SIGNAL_LBBB.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)


# ======================================================================================
# 15. SAVE FINAL ARRAY
# ======================================================================================

if not np.all(
    np.isfinite(
        X_final
    )
):
    raise RuntimeError(
        "Final array contains non-finite values."
    )

FINAL_PATH = (
    OUT_DIR
    /
    "X_gaze_final.npy"
)

np.save(
    FINAL_PATH,
    X_final
)


# ======================================================================================
# 16. HASHES / METADATA
# ======================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


parent_hash = sha256_file(
    PARENT_PATH
)

final_hash = sha256_file(
    FINAL_PATH
)

metadata = {

    "created":
        datetime.now().isoformat(),

    "parent_dataset":
        str(
            PARENT_PATH
        ),

    "parent_sha256":
        parent_hash,

    "final_LBBB_annotation_json":
        str(
            ANNOTATION_PATH
        ),

    "final_dataset":
        str(
            FINAL_PATH
        ),

    "final_sha256":
        final_hash,

    "shape":
        list(
            X_final.shape
        ),

    "final_replaced_leads": [
        "LBBB V1",
        "LBBB V2",
        "LBBB V3",
    ],

    "method": (
        "Final targeted replacement of LBBB V1-V3 using "
        "asymmetrically expanded source-image crops, manually placed "
        "waveform anchors, shape-preserving PCHIP interpolation and "
        "local image snapping of no more than six vertical pixels. "
        "All remaining V7 pseudo-waveforms were retained exactly."
    ),

    "model_training_performed":
        False,
}

with open(
    OUT_DIR
    /
    "metadata.json",
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )


# ======================================================================================
# 17. FINISH
# ======================================================================================

print("\n" + "=" * 100)
print("V8 FINAL PATCH COMPLETE")
print("=" * 100)

print(
    "\nFinal candidate dataset:"
)

print(
    FINAL_PATH
)

print(
    "\nShape:",
    X_final.shape
)

print(
    "\nV7 parent SHA256:"
)

print(
    parent_hash
)

print(
    "\nV8 final SHA256:"
)

print(
    final_hash
)

print(
    """
NOW SEND ME:

1. final_digitisation_v8/FINAL_OVERLAY_LBBB.png

2. The printed table headed:

       FINAL LBBB EDGE QC

If V1, V2 and V3 now show complete QRS complexes with no genuine
crop-boundary clipping, we freeze:

       final_digitisation_v8/X_gaze_final.npy

and move directly to the definitive clean gaze-training experiment.
"""
)

In [ ]:
# ======================================================================================
# FINAL DIGITISATION PATCH — V9
#
# Parent:
#     final_digitisation_v8/X_gaze_final.npy
#
# New annotation:
#     LOCAL_USER_HOME/Downloads/ecg_anchor_annotations_LBBB_V2_final*.json
#
# Replaces ONLY:
#     LBBB V2
#
# Output:
#     final_digitisation_v9/
#         X_gaze_final.npy
#         FINAL_OVERLAY_LBBB_V2.png
#         FINAL_SIGNAL_LBBB.png
#         final_LBBB_V2_metrics.csv
#         metadata.json
#
# DO NOT START MODEL TRAINING UNTIL THIS FINAL OVERLAY IS CHECKED.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import json
import hashlib
import warnings

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from scipy.ndimage import gaussian_filter
from scipy.interpolate import PchipInterpolator


warnings.filterwarnings("ignore")


# ======================================================================================
# 1. PATHS
# ======================================================================================

OUT_DIR = Path(
    "final_digitisation_v9"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PARENT_PATH = Path(
    "final_digitisation_v8/X_gaze_final.npy"
)


GEOMETRY_PATH = Path(
    "manual_digitization_qc/manual_geometry.json"
)


DOWNLOADS = Path(
    "LOCAL_USER_HOME/Downloads"
)


json_candidates = list(
    DOWNLOADS.glob(
        "ecg_anchor_annotations_LBBB_V2_final*.json"
    )
)


if not json_candidates:

    raise FileNotFoundError(
        "Could not find "
        "ecg_anchor_annotations_LBBB_V2_final*.json "
        "in LOCAL_USER_HOME/Downloads/"
    )


ANNOTATION_PATH = max(
    json_candidates,
    key=lambda p: p.stat().st_mtime
)


for p in [
    PARENT_PATH,
    GEOMETRY_PATH,
    ANNOTATION_PATH
]:

    if not p.exists():

        raise FileNotFoundError(
            p
        )


print("=" * 100)
print("FINAL V9 INPUTS")
print("=" * 100)

print(
    "Parent V8:",
    PARENT_PATH
)

print(
    "Geometry: ",
    GEOMETRY_PATH
)

print(
    "V2 JSON:  ",
    ANNOTATION_PATH
)


# ======================================================================================
# 2. LOAD
# ======================================================================================

X_v8 = np.load(
    PARENT_PATH
).astype(
    np.float32
)


if X_v8.shape != (
    10,
    12,
    1000,
    1
):

    raise ValueError(
        f"Unexpected parent array shape: {X_v8.shape}"
    )


X_final = X_v8.copy()


with open(
    GEOMETRY_PATH,
    "r"
) as f:

    geometry = json.load(
        f
    )


with open(
    ANNOTATION_PATH,
    "r"
) as f:

    annotation_json = json.load(
        f
    )


annotations = annotation_json.get(
    "annotations",
    {}
)


KEY = "LBBB::V2"


if KEY not in annotations:

    raise RuntimeError(
        f"{KEY} not found in annotation JSON."
    )


ann = annotations[
    KEY
]


# ======================================================================================
# 3. FIXED AXIS ORDERS
# ======================================================================================

ECG_ORDER = [

    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


LEAD_ORDER = [

    "I",
    "II",
    "III",

    "aVR",
    "aVL",
    "aVF",

    "V1",
    "V2",
    "V3",

    "V4",
    "V5",
    "V6",
]


LBBB_IDX = ECG_ORDER.index(
    "LBBB"
)


V2_IDX = LEAD_ORDER.index(
    "V2"
)


# ======================================================================================
# 4. VERIFY FINAL ANNOTATION
# ======================================================================================

status = ann.get(
    "status"
)


n_points = len(
    ann.get(
        "points",
        []
    )
)


print(
    "\n" +
    "=" * 100
)

print(
    "ANNOTATION CHECK"
)

print(
    "=" * 100
)


print(
    "Status:",
    status
)

print(
    "Anchors:",
    n_points
)


if status != "corrected":

    raise RuntimeError(
        "Final LBBB V2 annotation is not marked corrected."
    )


if n_points < 20:

    raise RuntimeError(
        f"Expected >=20 anchors, found {n_points}."
    )


print(
    "\n✓ Final LBBB V2 annotation is complete."
)


# ======================================================================================
# 5. SOURCE IMAGE HELPERS
# ======================================================================================

def resolve_source_image(
    stored_path
):

    p = Path(
        stored_path
    )


    if p.exists():

        return p


    matches = list(
        Path(".").glob(
            f"**/{p.name}"
        )
    )


    if not matches:

        raise FileNotFoundError(
            f"Could not locate source ECG image: {p.name}"
        )


    return matches[
        0
    ]


def load_gray(
    path
):

    return (
        np.asarray(
            Image.open(
                path
            )
            .convert(
                "L"
            )
        )
        .astype(
            np.float32
        )
        /
        255.0
    )


# ======================================================================================
# 6. MAP BROWSER POINTS TO ORIGINAL ULTRA-DEEP CROP
# ======================================================================================

def browser_points_to_crop(
    ann
):

    points = np.asarray(

        [
            [
                float(
                    p["x"]
                ),
                float(
                    p["y"]
                )
            ]

            for p in ann[
                "points"
            ]
        ],

        dtype=np.float32
    )


    display_w = float(
        ann[
            "display_width"
        ]
    )


    display_h = float(
        ann[
            "display_height"
        ]
    )


    original_w = float(
        ann[
            "original_crop_width"
        ]
    )


    original_h = float(
        ann[
            "original_crop_height"
        ]
    )


    points[
        :,
        0
    ] *= (
        original_w
        /
        display_w
    )


    points[
        :,
        1
    ] *= (
        original_h
        /
        display_h
    )


    points = points[
        np.argsort(
            points[
                :,
                0
            ]
        )
    ]


    # Remove near-duplicate x coordinates
    cleaned = [
        points[
            0
        ]
    ]


    for p in points[
        1:
    ]:

        if (
            p[
                0
            ]
            -
            cleaned[
                -1
            ][
                0
            ]
            >=
            0.5
        ):

            cleaned.append(
                p
            )


    points = np.asarray(
        cleaned,
        dtype=np.float32
    )


    if len(
        points
    ) < 18:

        raise RuntimeError(
            "Too few unique anchors after coordinate conversion."
        )


    return points


# ======================================================================================
# 7. ECG INK EVIDENCE MAP
# ======================================================================================

def ecg_score_map(
    gray
):

    gray = np.asarray(
        gray,
        dtype=np.float32
    )


    p1 = np.percentile(
        gray,
        1
    )


    p99 = np.percentile(
        gray,
        99
    )


    if p99 > p1 + 1e-8:

        norm = (
            gray
            -
            p1
        ) / (
            p99
            -
            p1
        )

    else:

        norm = gray.copy()


    norm = np.clip(
        norm,
        0,
        1
    )


    darkness = (
        1.0
        -
        norm
    )


    fine = gaussian_filter(
        norm,
        sigma=0.45
    )


    background = gaussian_filter(
        norm,
        sigma=2.2
    )


    ridge = np.clip(
        background
        -
        fine,
        0,
        None
    )


    ridge_scale = np.percentile(
        ridge,
        99.5
    )


    if ridge_scale > 1e-8:

        ridge /= ridge_scale


    ridge = np.clip(
        ridge,
        0,
        1.5
    )


    score = (
        0.75
        *
        ridge
        +
        0.25
        *
        darkness
    )


    score_scale = np.percentile(
        score,
        99.5
    )


    if score_scale > 1e-8:

        score /= score_scale


    return np.clip(
        score,
        0,
        1.5
    )


# ======================================================================================
# 8. FULL-WIDTH MANUAL GUIDE
# ======================================================================================

def make_full_width_guide(
    anchors,
    width
):

    xs = np.arange(
        width,
        dtype=np.float32
    )


    first_x = float(
        anchors[
            0,
            0
        ]
    )


    last_x = float(
        anchors[
            -1,
            0
        ]
    )


    first_y = float(
        anchors[
            0,
            1
        ]
    )


    last_y = float(
        anchors[
            -1,
            1
        ]
    )


    interpolator = PchipInterpolator(

        anchors[
            :,
            0
        ],

        anchors[
            :,
            1
        ],

        extrapolate=False
    )


    guide = np.empty(
        width,
        dtype=np.float32
    )


    inside = (
        (
            xs
            >=
            first_x
        )
        &
        (
            xs
            <=
            last_x
        )
    )


    guide[
        inside
    ] = interpolator(
        xs[
            inside
        ]
    ).astype(
        np.float32
    )


    # Preserve full horizontal duration.
    guide[
        xs
        <
        first_x
    ] = first_y


    guide[
        xs
        >
        last_x
    ] = last_y


    return (
        xs,
        guide
    )


# ======================================================================================
# 9. FINAL ONE-LEAD RECONSTRUCTION
# ======================================================================================

def reconstruct_final_v2(
    crop,
    anchors
):

    H, W = crop.shape


    anchors = np.asarray(
        anchors,
        dtype=np.float32
    )


    anchors[
        :,
        0
    ] = np.clip(
        anchors[
            :,
            0
        ],
        0,
        W - 1
    )


    anchors[
        :,
        1
    ] = np.clip(
        anchors[
            :,
            1
        ],
        0,
        H - 1
    )


    first_fraction = (
        anchors[
            0,
            0
        ]
        /
        max(
            W - 1,
            1
        )
    )


    last_fraction = (
        anchors[
            -1,
            0
        ]
        /
        max(
            W - 1,
            1
        )
    )


    span_fraction = (
        anchors[
            -1,
            0
        ]
        -
        anchors[
            0,
            0
        ]
    ) / max(
        W - 1,
        1
    )


    # Final annotator required stricter coverage.
    if first_fraction > 0.09:

        raise RuntimeError(
            f"First anchor too far from left edge: {first_fraction:.3f}"
        )


    if last_fraction < 0.91:

        raise RuntimeError(
            f"Last anchor too far from right edge: {last_fraction:.3f}"
        )


    if span_fraction < 0.82:

        raise RuntimeError(
            f"Insufficient horizontal coverage: {span_fraction:.3f}"
        )


    xs, guide = make_full_width_guide(
        anchors,
        W
    )


    guide = np.clip(
        guide,
        0,
        H - 1
    )


    score = ecg_score_map(
        crop
    )


    # The manual path remains dominant.
    snap_radius = min(
        6,
        max(
            3,
            int(
                round(
                    0.025
                    *
                    H
                )
            )
        )
    )


    path = np.zeros(
        W,
        dtype=np.float32
    )


    first_x = float(
        anchors[
            0,
            0
        ]
    )


    last_x = float(
        anchors[
            -1,
            0
        ]
    )


    previous_y = None


    for x in range(
        W
    ):

        gy = float(
            guide[
                x
            ]
        )


        centre = int(
            round(
                gy
            )
        )


        if (
            x
            <
            first_x
            or
            x
            >
            last_x
        ):

            radius = min(
                3,
                snap_radius
            )

            distance_weight = 0.35

        else:

            radius = snap_radius

            distance_weight = 0.22


        lo = max(
            0,
            centre
            -
            radius
        )


        hi = min(
            H - 1,
            centre
            +
            radius
        )


        ys = np.arange(
            lo,
            hi + 1,
            dtype=int
        )


        objective = (

            2.0
            *
            score[
                ys,
                x
            ]

            -

            distance_weight
            *
            np.abs(
                ys
                -
                gy
            )
        )


        # Very weak continuity preference.
        if previous_y is not None:

            objective -= (
                0.015
                *
                np.abs(
                    ys
                    -
                    previous_y
                )
            )


        best_y = ys[
            np.argmax(
                objective
            )
        ]


        path[
            x
        ] = float(
            best_y
        )


        previous_y = float(
            best_y
        )


    # Exact agreement at manually annotated x positions.
    for ax, ay in anchors:

        xi = int(
            np.clip(
                round(
                    ax
                ),
                0,
                W - 1
            )
        )


        path[
            xi
        ] = float(
            ay
        )


    # ==================================================================================
    # ECG SIGNAL
    # ==================================================================================

    signal = (
        -path
    )


    signal -= np.median(
        signal
    )


    signal_1000 = np.interp(

        np.linspace(
            0,
            1,
            1000
        ),

        np.linspace(
            0,
            1,
            W
        ),

        signal

    ).astype(
        np.float32
    )


    signal_1000 -= np.mean(
        signal_1000
    )


    sd = np.std(
        signal_1000
    )


    if sd > 1e-8:

        signal_1000 /= sd


    # ==================================================================================
    # FINAL EDGE QC
    # ==================================================================================

    distance_to_top = path


    distance_to_bottom = (
        H
        -
        1
        -
        path
    )


    distance_to_edge = np.minimum(
        distance_to_top,
        distance_to_bottom
    )


    # Same definition used in previous QC
    edge_margin = max(
        3,
        int(
            round(
                0.02
                *
                H
            )
        )
    )


    top_contact = (
        path
        <=
        edge_margin
    )


    bottom_contact = (
        path
        >=
        H
        -
        1
        -
        edge_margin
    )


    edge_contact = (
        top_contact
        |
        bottom_contact
    )


    longest_run = 0
    current_run = 0


    for contact in edge_contact:

        if contact:

            current_run += 1

            longest_run = max(
                longest_run,
                current_run
            )

        else:

            current_run = 0


    deviation = np.abs(
        path
        -
        guide
    )


    y_indices = np.clip(
        np.round(
            path
        ).astype(
            int
        ),
        0,
        H - 1
    )


    path_scores = score[
        y_indices,
        np.arange(
            W
        )
    ]


    metrics = {

        "n_anchors":
            int(
                len(
                    anchors
                )
            ),

        "crop_width":
            int(
                W
            ),

        "crop_height":
            int(
                H
            ),

        "first_anchor_fraction":
            float(
                first_fraction
            ),

        "last_anchor_fraction":
            float(
                last_fraction
            ),

        "annotation_span_fraction":
            float(
                span_fraction
            ),

        "reconstruction_span_fraction":
            1.0,

        "snap_radius_px":
            int(
                snap_radius
            ),

        "median_snap_distance_px":
            float(
                np.median(
                    deviation
                )
            ),

        "p95_snap_distance_px":
            float(
                np.percentile(
                    deviation,
                    95
                )
            ),

        "minimum_distance_to_edge_px":
            float(
                np.min(
                    distance_to_edge
                )
            ),

        "minimum_distance_to_bottom_px":
            float(
                np.min(
                    distance_to_bottom
                )
            ),

        "top_edge_contact_fraction":
            float(
                np.mean(
                    top_contact
                )
            ),

        "bottom_edge_contact_fraction":
            float(
                np.mean(
                    bottom_contact
                )
            ),

        "any_edge_contact_fraction":
            float(
                np.mean(
                    edge_contact
                )
            ),

        "longest_edge_contact_run_px":
            int(
                longest_run
            ),

        "mean_path_score":
            float(
                np.mean(
                    path_scores
                )
            ),

        "median_path_score":
            float(
                np.median(
                    path_scores
                )
            ),
    }


    return {

        "x":
            np.arange(
                W,
                dtype=np.float32
            ),

        "y":
            path.astype(
                np.float32
            ),

        "guide":
            guide.astype(
                np.float32
            ),

        "anchors":
            anchors.astype(
                np.float32
            ),

        "signal":
            signal_1000.astype(
                np.float32
            ),

        "metrics":
            metrics,
    }


# ======================================================================================
# 10. LOAD ULTRA-DEEP SOURCE CROP
# ======================================================================================

source_path = resolve_source_image(
    geometry[
        "LBBB"
    ][
        "image_path"
    ]
)


source_gray = load_gray(
    source_path
)


box = [
    int(
        round(
            value
        )
    )
    for value
    in ann[
        "original_box"
    ]
]


x0, y0, x1, y1 = (
    box
)


crop = source_gray[
    y0:y1,
    x0:x1
]


anchors = browser_points_to_crop(
    ann
)


print(
    "\nFinal ultra-deep crop shape:",
    crop.shape
)


print(
    "Unique manual anchors:",
    len(
        anchors
    )
)


# ======================================================================================
# 11. RECONSTRUCT LBBB V2
# ======================================================================================

result = reconstruct_final_v2(
    crop=crop,
    anchors=anchors
)


# Replace ONLY LBBB V2.
X_final[
    LBBB_IDX,
    V2_IDX,
    :,
    0
] = result[
    "signal"
]


# ======================================================================================
# 12. PROVE ONLY ONE LEAD CHANGED
# ======================================================================================

changed = []


for ei, ecg in enumerate(
    ECG_ORDER
):

    for li, lead in enumerate(
        LEAD_ORDER
    ):

        max_diff = float(
            np.max(
                np.abs(
                    X_final[
                        ei,
                        li,
                        :,
                        0
                    ]
                    -
                    X_v8[
                        ei,
                        li,
                        :,
                        0
                    ]
                )
            )
        )


        if max_diff > 1e-7:

            changed.append(
                {
                    "ECG":
                        ecg,

                    "lead":
                        lead,

                    "max_abs_change":
                        max_diff
                }
            )


changed_df = pd.DataFrame(
    changed
)


print(
    "\n" +
    "=" * 100
)

print(
    "CHANGED LEADS VS V8"
)

print(
    "=" * 100
)


print(
    changed_df.to_string(
        index=False
    )
)


actual_changed = {
    (
        row[
            "ECG"
        ],
        row[
            "lead"
        ]
    )
    for row
    in changed
}


expected_changed = {
    (
        "LBBB",
        "V2"
    )
}


if actual_changed != expected_changed:

    raise RuntimeError(
        "\nUnexpected changes detected.\n"
        f"Expected: {expected_changed}\n"
        f"Actual:   {actual_changed}"
    )


print(
    "\n✓ Exactly ONE signal changed: LBBB V2."
)

print(
    "✓ The other 119 ECG-lead waveforms are identical to V8."
)


# ======================================================================================
# 13. FINAL QC TABLE
# ======================================================================================

metrics_df = pd.DataFrame(
    [
        {
            "ECG":
                "LBBB",

            "lead":
                "V2",

            **result[
                "metrics"
            ]
        }
    ]
)


metrics_df[
    "final_edge_flag"
] = (

    (
        metrics_df[
            "any_edge_contact_fraction"
        ]
        >
        0.01
    )

    |

    (
        metrics_df[
            "minimum_distance_to_edge_px"
        ]
        <
        3
    )

    |

    (
        metrics_df[
            "longest_edge_contact_run_px"
        ]
        >
        5
    )
)


metrics_df.to_csv(

    OUT_DIR
    /
    "final_LBBB_V2_metrics.csv",

    index=False
)


print(
    "\n" +
    "=" * 100
)

print(
    "FINAL LBBB V2 EDGE QC"
)

print(
    "=" * 100
)


print(
    metrics_df[
        [
            "lead",
            "n_anchors",
            "crop_height",
            "first_anchor_fraction",
            "last_anchor_fraction",
            "annotation_span_fraction",
            "minimum_distance_to_edge_px",
            "minimum_distance_to_bottom_px",
            "top_edge_contact_fraction",
            "bottom_edge_contact_fraction",
            "any_edge_contact_fraction",
            "longest_edge_contact_run_px",
            "final_edge_flag",
        ]
    ]
    .to_string(
        index=False
    )
)


# ======================================================================================
# 14. FINAL V2 OVERLAY
# ======================================================================================

fig, axes = plt.subplots(

    1,
    2,

    figsize=(
        18,
        6
    ),

    constrained_layout=True
)


# --------------------------------------------------------------------------------------
# Source + path
# --------------------------------------------------------------------------------------

ax = axes[
    0
]


ax.imshow(
    crop,
    cmap="gray",
    origin="upper",
    aspect="auto"
)


ax.plot(
    result[
        "x"
    ],
    result[
        "guide"
    ],
    linestyle="--",
    linewidth=1.0,
    label="manual full-width guide"
)


ax.plot(
    result[
        "x"
    ],
    result[
        "y"
    ],
    color="red",
    linewidth=1.6,
    label="final path"
)


ax.scatter(
    result[
        "anchors"
    ][
        :,
        0
    ],
    result[
        "anchors"
    ][
        :,
        1
    ],
    s=20,
    facecolors="none",
    edgecolors="yellow",
    linewidths=1,
    label="manual anchors"
)


ax.set_xlim(
    0,
    crop.shape[
        1
    ]
    -
    1
)


ax.set_ylim(
    crop.shape[
        0
    ]
    -
    1,
    0
)


ax.set_title(
    "LBBB V2: FINAL ultra-deep source + path"
)


ax.set_xticks([])
ax.set_yticks([])


ax.legend(
    fontsize=8,
    loc="upper right"
)


# --------------------------------------------------------------------------------------
# Final pseudo-waveform
# --------------------------------------------------------------------------------------

ax = axes[
    1
]


ax.plot(
    X_final[
        LBBB_IDX,
        V2_IDX,
        :,
        0
    ],
    linewidth=1.0
)


ax.axhline(
    0,
    linewidth=0.4,
    alpha=0.3
)


ax.set_xlim(
    0,
    999
)


ax.set_title(
    "LBBB V2: FINAL full-duration pseudo-waveform"
)


ax.set_xticks([])
ax.set_yticks([])


fig.suptitle(
    "LBBB V2 — ABSOLUTE FINAL DIGITISATION QC",
    fontsize=18
)


fig.savefig(

    OUT_DIR
    /
    "FINAL_OVERLAY_LBBB_V2.png",

    dpi=220,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ======================================================================================
# 15. FINAL LBBB 12-LEAD SHEET
# ======================================================================================

DISPLAY_GRID = [

    ["I",   "aVR", "V1", "V4"],

    ["II",  "aVL", "V2", "V5"],

    ["III", "aVF", "V3", "V6"],
]


fig, axes = plt.subplots(

    3,
    4,

    figsize=(
        18,
        10
    ),

    constrained_layout=True
)


for r in range(
    3
):

    for c in range(
        4
    ):

        lead = DISPLAY_GRID[
            r
        ][
            c
        ]


        li = LEAD_ORDER.index(
            lead
        )


        sig = X_final[
            LBBB_IDX,
            li,
            :,
            0
        ]


        ax = axes[
            r,
            c
        ]


        ax.plot(
            sig,
            linewidth=1
        )


        ax.axhline(
            0,
            linewidth=0.4,
            alpha=0.3
        )


        if lead == "V2":

            suffix = (
                " [V9 FINAL]"
            )

        elif lead in [
            "V1",
            "V3"
        ]:

            suffix = (
                " [V8 FINAL]"
            )

        else:

            suffix = (
                " [RETAINED]"
            )


        ax.set_title(
            lead
            +
            suffix
        )


        ax.set_xlim(
            0,
            999
        )


        ax.set_xticks([])
        ax.set_yticks([])


fig.suptitle(
    "LBBB — FINAL V9 12-lead candidate",
    fontsize=18
)


fig.savefig(

    OUT_DIR
    /
    "FINAL_SIGNAL_LBBB.png",

    dpi=200,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ======================================================================================
# 16. SAVE FINAL DATASET
# ======================================================================================

if not np.all(
    np.isfinite(
        X_final
    )
):

    raise RuntimeError(
        "Non-finite values detected in final dataset."
    )


FINAL_PATH = (
    OUT_DIR
    /
    "X_gaze_final.npy"
)


np.save(
    FINAL_PATH,
    X_final
)


# ======================================================================================
# 17. HASHES
# ======================================================================================

def sha256_file(
    path
):

    h = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024
                *
                1024
            )


            if not block:

                break


            h.update(
                block
            )


    return h.hexdigest()


parent_hash = sha256_file(
    PARENT_PATH
)


final_hash = sha256_file(
    FINAL_PATH
)


# ======================================================================================
# 18. METADATA
# ======================================================================================

metadata = {

    "created":
        datetime.now().isoformat(),

    "parent_dataset":
        str(
            PARENT_PATH
        ),

    "parent_sha256":
        parent_hash,

    "final_annotation_json":
        str(
            ANNOTATION_PATH
        ),

    "final_dataset":
        str(
            FINAL_PATH
        ),

    "final_sha256":
        final_hash,

    "shape":
        list(
            X_final.shape
        ),

    "ecg_order":
        ECG_ORDER,

    "lead_order":
        LEAD_ORDER,

    "final_replaced_signal":
        "LBBB V2",

    "number_signals_changed_from_v8":
        1,

    "method": (
        "Final replacement of LBBB V2 using an ultra-deep asymmetric "
        "source-image crop. Dense manually placed waveform anchors defined "
        "a shape-preserving PCHIP trajectory across the diagnostic segment, "
        "with local image snapping restricted to at most six vertical pixels. "
        "The complete crop width was mapped to 1000 samples to preserve "
        "horizontal segment duration. All other 119 ECG-lead pseudo-waveforms "
        "were retained exactly from the V8 parent dataset."
    ),

    "model_training_performed":
        False,
}


with open(

    OUT_DIR
    /
    "metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )


# ======================================================================================
# 19. FINAL REPORT
# ======================================================================================

print(
    "\n" +
    "=" * 100
)

print(
    "V9 FINAL ONE-LEAD PATCH COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nCandidate final dataset:"
)

print(
    FINAL_PATH
)


print(
    "\nShape:"
)

print(
    X_final.shape
)


print(
    "\nParent V8 SHA256:"
)

print(
    parent_hash
)


print(
    "\nCandidate V9 SHA256:"
)

print(
    final_hash
)


print(
    """
NOW SEND ME ONLY TWO THINGS:

1. The image:

       final_digitisation_v9/FINAL_OVERLAY_LBBB_V2.png

2. The printed table:

       FINAL LBBB V2 EDGE QC

If the waveform is visibly complete and the numerical QC confirms that
the deep QRS no longer reaches the crop boundary, we freeze:

       final_digitisation_v9/X_gaze_final.npy

No further ECG digitisation iterations after that.
"""
)